## Imports  

#### Loads the libraries we use: json to read the data, re and string for cleaning text, Counter for counting words, BeautifulSoup to remove HTML, sklearn for the classifiers and nltk for bigrams.

In [ ]:
import pandas as pd
import json
import re
import string
import math
import numpy as np
from collections import Counter, defaultdict
from bs4 import BeautifulSoup
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from nltk import bigrams
import spacy
import pickle
import os
import time
from sklearn.feature_extraction import DictVectorizer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score
from scipy.sparse import hstack, csr_matrix
import gensim.downloader as api
from gensim.models import Word2Vec
import torch
import random
from torch.utils.data import Dataset, DataLoader
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          AutoModelForMaskedLM, DataCollatorWithPadding, pipeline)

## Load Data

#### The data is in JSONL format, so every line is one JSON object (one text). load_jsonl reads the file line by line and puts each text into a list. We load train.jsonl and dev.jsonl.

In [7]:
def load_jsonl(path):
    data = []
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            record = json.loads(line)
            data.append(record)
    return data

train = load_jsonl('train.jsonl')
dev = load_jsonl('dev.jsonl')

## Explore Data

#### We look at the data before doing anything. Each text has an id, text, model, label and genre. Label 0 = human, label 1 = AI.
- Train has 23,707 texts and dev has 1,795.
- About 62% of train is AI, so the data is imbalanced.
- Fiction is about 50/50, but essays and news are about 80% AI.
- Human texts are longer on average (695 words vs 575) and vary more in length.

In [8]:
print(len(train), len(dev))
print(train[0].keys())
print(train[0])
print(Counter(r['label'] for r in train))
print(Counter(r['label'] for r in dev))
print(Counter((r['label'], r['model']) for r in train))
print(Counter(r['genre'] for r in train))
print(Counter((r['label'], r['genre']) for r in train))
for label in set(r['label'] for r in train):
    lengths = [len(r['text'].split()) for r in train if r['label'] == label]
    print(label, 'min:', min(lengths), 'max:', max(lengths), 'gns:', round(sum(lengths)/len(lengths)))

23707 1795
dict_keys(['id', 'text', 'model', 'label', 'genre'])
{'id': 'ea468d03-1973-5039-86b2-ff225bb92c4e', 'text': 'Duke Ellington, a titan of jazz, revolutionized the genre through his innovative compositions, showcasing a remarkable ability to integrate voice and instrumental music. Among the notable figures who contributed to this artistic symphony was Ivie Anderson, whose scat singing mirrored the improvisational prowess of musicians like Nanton. In "Ring Dem Bells," Ellington ingeniously interweaves scat singing as a dialogue with saxophones, creating a dynamic call and response that underscores his vision of music as a fluid conversation between voices and instruments.\n\nEllington consistently emphasized the voice as an instrument of equal importance to traditional brass and woodwinds, orchestrating his compositions with a keen ear for vocal qualities. This approach is vividly demonstrated in "Mood Indigo," where his orchestration skills shine through non-traditional chord a

## Pre-processing

#### clean_text normalizes the text: lowercase, removes HTML, URLs, numbers and punctuation, and removes extra spaces. Tokenize splits the cleaned text into words on spaces. We save the tokens for every text in r['tokens'] so we don't have to clean again later.

In [9]:
def clean_text(text):
    text = text.lower()
    text = BeautifulSoup(text, "html.parser").get_text()
    text = re.sub(r'http\S+|www\S+', '', text)
    text = re.sub(r'\d+', '', text)
    text = text.translate(str.maketrans('', '', string.punctuation))
    text = re.sub(r'\W+', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

In [10]:
print(train[0]['text'][:300])
print()
print(clean_text(train[0]['text'])[:300])

Duke Ellington, a titan of jazz, revolutionized the genre through his innovative compositions, showcasing a remarkable ability to integrate voice and instrumental music. Among the notable figures who contributed to this artistic symphony was Ivie Anderson, whose scat singing mirrored the improvisati

duke ellington a titan of jazz revolutionized the genre through his innovative compositions showcasing a remarkable ability to integrate voice and instrumental music among the notable figures who contributed to this artistic symphony was ivie anderson whose scat singing mirrored the improvisational 


In [11]:
def tokenize(text):
    return clean_text(text).split()

print(tokenize(train[0]['text'])[:30])

['duke', 'ellington', 'a', 'titan', 'of', 'jazz', 'revolutionized', 'the', 'genre', 'through', 'his', 'innovative', 'compositions', 'showcasing', 'a', 'remarkable', 'ability', 'to', 'integrate', 'voice', 'and', 'instrumental', 'music', 'among', 'the', 'notable', 'figures', 'who', 'contributed', 'to']


In [12]:
for r in train:
    r['tokens'] = tokenize(r['text'])
for r in dev:
    r['tokens'] = tokenize(r['text'])

print(train[0]['tokens'][:30])

['duke', 'ellington', 'a', 'titan', 'of', 'jazz', 'revolutionized', 'the', 'genre', 'through', 'his', 'innovative', 'compositions', 'showcasing', 'a', 'remarkable', 'ability', 'to', 'integrate', 'voice', 'and', 'instrumental', 'music', 'among', 'the', 'notable', 'figures', 'who', 'contributed', 'to']


## Vocabulary

#### I count words separately for human and AI texts with Counter. Humans have 112,763 unique words and AI has 79,963, so humans use a more varied vocabulary. The most common words are function words like "the", "and" and "of".

I also found 281 texts with Chinese characters. All except one are written by AI (mostly deepseek). A model could learn "Chinese = AI" as a shortcut.

72.5% of the unique words in train never show up in dev. Most of them are rare words like names and typos.

In [13]:
human_Vocab = Counter()
machine_Vocab = Counter()

for r in train:
    if r['label'] == 0:
        human_Vocab.update(r['tokens'])
    else:
        machine_Vocab.update(r['tokens'])

print('Human:', len(human_Vocab), 'unique ord')
print('Machine:', len(machine_Vocab), 'unique ord')

Human: 112763 unique ord
Machine: 79963 unique ord


In [14]:
print(human_Vocab.most_common(30))
print()
print(machine_Vocab.most_common(30))

[('the', 357452), ('and', 198040), ('of', 184763), ('to', 173344), ('a', 152700), ('in', 109274), ('was', 83214), ('i', 81013), ('that', 78351), ('he', 78187), ('it', 68253), ('his', 63385), ('with', 52809), ('for', 51641), ('her', 51619), ('as', 51072), ('you', 50897), ('had', 48441), ('is', 43679), ('she', 41618), ('not', 38855), ('on', 38624), ('at', 38526), ('but', 37344), ('be', 34163), ('have', 30589), ('said', 27227), ('this', 26350), ('by', 25942), ('him', 25561)]

[('the', 575528), ('of', 325045), ('a', 260664), ('and', 218424), ('to', 195536), ('in', 141319), ('that', 100297), ('his', 99321), ('with', 95390), ('as', 81878), ('her', 77276), ('for', 64847), ('he', 56215), ('was', 54932), ('i', 49502), ('it', 45651), ('by', 41466), ('their', 41313), ('on', 40505), ('is', 40059), ('had', 36865), ('not', 35497), ('this', 34894), ('but', 34169), ('she', 33571), ('from', 31793), ('an', 29253), ('its', 26919), ('at', 26438), ('my', 24816)]


In [15]:
all_Vocab = human_Vocab + machine_Vocab
print(sorted(all_Vocab)[:50])
print(sorted(all_Vocab)[-50:])

['a', 'aa', 'aaa', 'aaaaaaaaaaaahahahahahahha', 'aaahouch', 'aaan', 'aaarh', 'aablest', 'aachen', 'aaeven', 'aahas', 'aahing', 'aaisnt', 'aal', 'aalamat', 'aan', 'aap', 'aappendix', 'aaqii', 'aaraaf', 'aarhus', 'aaron', 'aaronever', 'aarons', 'aashe', 'aasheim', 'aasm', 'aautoloadfächerous', 'ab', 'aba', 'aback', 'abacterial', 'abacus', 'abaft', 'abandon', 'abandoned', 'abandoneda', 'abandonedof', 'abandonedthe', 'abandonedwhat', 'abandoning', 'abandonment', 'abandons', 'abapical', 'abase', 'abasement', 'abashd', 'abashed', 'abashedly', 'abasin']
['这可不像小事', '这是他熟悉的土地', '这样的行为不会被容忍', '这还没完', '选择坚守我们的岗位', '遮蔽了天空', '那一刻', '那些为了鸡毛蒜皮的小事而大动肝火的日子', '那些人仿佛都被时间带走了', '那些建筑虽然依旧矗立', '那些无谓的争执', '那些还记得', '那些陡峭的峡湾和高耸的山脉仿佛触手可及', '那双深邃的眼睛透着岁月的沧桑与智慧的光芒', '那或许是曾经的某个熟人', '那时的他们', '那时的他们常常在月光下讲述自己的故事', '那时的他还年轻', '那时的海面被月光覆盖', '那是他在海上多年的伴侣', '那男子大声问道', '那艘船', '那艘船早已被人遗忘', '那里却矗立着一座新的高楼', '那里如今已经换上了新的招牌', '那里如今摆满了他从未见过的新鲜海鲜', '那里或许还残留着一些他记忆中的影子', '那里曾经是他和同伴们常常聚会的地方', '都似乎在这一次的摔打之后烟消云散了', '银色的头发在午后斜阳中泛着微光', 

In [16]:
Chinese = [r for r in train if re.search(r'[\u4e00-\u9fff]', r['text'])]
print(len(Chinese))
print(Counter((r['label'], r['model']) for r in Chinese))

281
Counter({(1, 'deepseek-r1-distill-qwen-32b'): 215, (1, 'ministral-8b-instruct-2410'): 39, (1, 'llama-3.1-8b-instruct'): 10, (1, 'qwen1.5-72b-chat-8bit'): 5, (1, 'gemini-1.5-pro'): 3, (1, 'text-bison-002'): 2, (1, 'gpt-4o-mini'): 2, (1, 'o3-mini'): 1, (1, 'gemini-2.0-flash'): 1, (1, 'gemini-pro-paraphrase'): 1, (0, 'human'): 1, (1, 'gemini-pro'): 1})


In [17]:
train_vocab = set(t for r in train for t in r['tokens'])
dev_vocab = set(t for r in dev for t in r['tokens'])

only_train = train_vocab - dev_vocab
print(round(len(only_train) / len(train_vocab) * 100, 1), '%')

72.5 %


## Question 4

**a) What kinds of words do you see?**
The top of the list is almost only function words like the, and, of, to. Humans use "i", "was" and "he" more, so they write more like a story in first person and past tense. The AI uses "of", "a", "with" and "as" more. In the alphabetical list there are also expressions like "aaaaahahaha" and "aarh", and at the end there is Chinese text.

**b) Does the text normalization work well?**
Mostly. Capital letters and punctuation are gone. But because punctuation is deleted instead of replaced with a space, some words get merged, like "aaeven". Special characters like ’ and – are not removed by string.punctuation, but the \W+ line after catches them and turns them into a space, so "don’t" becomes "don t". And Chinese is not split into words, because there are no spaces, so whole sentences become one token.

**c) Are there marked differences between human_Vocab and machine_Vocab?**
Yes. Humans have 112,763 unique words and the AI only 79,963, even though there are fewer human texts. The AI texts also have more tokens in total (about 14,606 × 575 ≈ 8.4M vs 9,101 × 695 ≈ 6.3M), so the AI writes more text but still uses fewer unique words. So humans have a more varied vocabulary.

**d) What % of tokens were seen in train but not dev?**
72.5%. Most of them are rare words like names and typos, and dev is much smaller than train.

**e) Other observations**
- The dataset is imbalanced: about 62% AI and 38% human.
- Fiction is about 50/50, but essays and news are about 80% AI.
- Human texts are longer on average and vary more in length.
- 281 texts contain Chinese, and all except one are written by AI (mostly deepseek). For example line 272 in train.jsonl, where the model switches to Chinese in the middle of an English sentence. A model could learn "Chinese = AI" as a shortcut.

## Baseline

#### Before training a model we check what accuracy we get by always guessing AI. This gives 0.644, because 64% of dev is AI. A model has to beat this to be useful. Because the data is imbalanced, we also look at F1 and not only accuracy.

In [18]:
train_texts = [' '.join(r['tokens']) for r in train]
dev_texts = [' '.join(r['tokens']) for r in dev]
y_train = [r['label'] for r in train]
y_dev = [r['label'] for r in dev]

baseline = Counter(y_dev)[1] / len(y_dev)
print('Baseline accuracy (always guess AI):', round(baseline, 3))

Baseline accuracy (always guess AI): 0.644


## Bag of words + Logistic Regression

#### CountVectorizer turns each text into a vector where each column is a word and the value is how many times it appears (bag of words). We fit it only on train and use transform on dev, so words that only appear in dev are ignored. Otherwise it would be data contamination.

We changed token_pattern so words with one letter like "i" and "a" are kept, because they were different between humans and AI in lesson 1.

Logistic regression learns a weight for each word. Result: accuracy 0.987 and F1 0.98 for human and 0.99 for AI. Only 14 human texts were guessed as AI and 9 AI texts as human.

In [19]:
vectorizer = CountVectorizer(token_pattern=r"(?u)\b\w+\b")
X_train = vectorizer.fit_transform(train_texts)
X_dev = vectorizer.transform(dev_texts)
print('Shape:', X_train.shape)

model = LogisticRegression(max_iter=2000)
model.fit(X_train, y_train)
y_pred = model.predict(X_dev)

print('Accuracy:', round(accuracy_score(y_dev, y_pred), 3))
print('Confusion matrix:\n', confusion_matrix(y_dev, y_pred))
print(classification_report(y_dev, y_pred, target_names=['human', 'AI']))

Shape: (23707, 139096)
Accuracy: 0.987
Confusion matrix:
 [[ 625   14]
 [   9 1147]]
              precision    recall  f1-score   support

       human       0.99      0.98      0.98       639
          AI       0.99      0.99      0.99      1156

    accuracy                           0.99      1795
   macro avg       0.99      0.99      0.99      1795
weighted avg       0.99      0.99      0.99      1795



#### We look at the weights the model learned. Words that point to AI are formal words like "significant", "despite", "crucial" and "however". Words that point to human are more like normal storytelling, like "because", "cried" and "wednesday".

In [20]:
words = vectorizer.get_feature_names_out()
coefs = model.coef_[0]

print('Most AI:', [words[i] for i in np.argsort(coefs)[-20:][::-1]])
print('Most human:', [words[i] for i in np.argsort(coefs)[:20]])

Most AI: ['significant', 'despite', 'crucial', 'however', 'concerns', 'challenges', 'particularly', 'investigation', 'individuals', 'remains', 'approach', 'confess', 'amidst', 'stated', 'emphasized', 'phebe', 'further', 'grandfather', 'expressed', 'incident']
Most human: ['because', 'cried', 'other', 'wednesday', 'although', 'two', 'fact', 'answered', 'says', 'few', 'month', 'gave', 'wife', 'having', 'does', 'wrote', 'scarcely', 'tuesday', 'order', 'pretty']


## TF-IDF

#### Same setup, but the vectors are TF-IDF instead of counts. TF-IDF gives low weight to words that are in almost every text and high weight to rare words. It also normalizes for text length.

Result: accuracy 0.969, so a bit worse than bag of words. The difference is mostly human recall (0.93 vs 0.98). Maybe because TF-IDF gives low weight to function words, which are actually useful for telling human and AI apart, and because it removes the length information.

The results are very high, but dev comes from the same AI models and genres as train, so it would probably be lower on text from a new AI model.

In [21]:
tfidf = TfidfVectorizer(token_pattern=r"(?u)\b\w+\b")
X_train_tfidf = tfidf.fit_transform(train_texts)
X_dev_tfidf = tfidf.transform(dev_texts)

model_tfidf = LogisticRegression(max_iter=2000)
model_tfidf.fit(X_train_tfidf, y_train)
y_pred_tfidf = model_tfidf.predict(X_dev_tfidf)

print('Accuracy:', round(accuracy_score(y_dev, y_pred_tfidf), 3))
print('Confusion matrix:\n', confusion_matrix(y_dev, y_pred_tfidf))
print(classification_report(y_dev, y_pred_tfidf, target_names=['human', 'AI']))

Accuracy: 0.969
Confusion matrix:
 [[ 597   42]
 [  14 1142]]
              precision    recall  f1-score   support

       human       0.98      0.93      0.96       639
          AI       0.96      0.99      0.98      1156

    accuracy                           0.97      1795
   macro avg       0.97      0.96      0.97      1795
weighted avg       0.97      0.97      0.97      1795



## Bigram language models

We train one bigram model on human texts and one on AI texts. nltk.bigrams makes all word pairs and adds `<s>` and `</s>` at the start and end of each text. The model counts how often each word comes after another word.

To classify a dev text we compute its perplexity under both models. The model with the lowest perplexity is the least surprised by the text, so that is the guess. We use Laplace / add-k smoothing so bigrams we never saw in train don't get probability 0, and we add log probabilities instead of multiplying to avoid underflow.

Result: accuracy 0.916 for all three values of k (1, 0.1 and 0.01), so the smoothing value almost didn't matter. That is better than the baseline (0.644) but worse than bag of words (0.987). The confusion matrix shows the model leans towards AI: 132 of the 639 human texts were guessed as AI, but only 19 of 1156 AI texts were guessed as human (human recall about 0.79, AI recall about 0.98). Maybe because AI text is more predictable in general, and because the AI model is trained on more text, so it is less often surprised.

In [22]:
def build_model(token_lists):
    model = defaultdict(Counter)
    for toks in token_lists:
        for w1, w2 in bigrams(toks, pad_left=True, pad_right=True,
                              left_pad_symbol='<s>', right_pad_symbol='</s>'):
            model[w1][w2] += 1
    totals = {w1: sum(nexts.values()) for w1, nexts in model.items()}
    return model, totals

human_model, human_totals = build_model([r['tokens'] for r in train if r['label'] == 0])
ai_model, ai_totals = build_model([r['tokens'] for r in train if r['label'] == 1])

In [23]:
def perplexity(toks, model, totals, V, k=1):
    pairs = list(bigrams(toks, pad_left=True, pad_right=True,
                         left_pad_symbol='<s>', right_pad_symbol='</s>'))
    log_prob = 0
    for w1, w2 in pairs:
        count = model.get(w1, {}).get(w2, 0)
        p = (count + k) / (totals.get(w1, 0) + k * V)
        log_prob += math.log(p)
    return math.exp(-log_prob / len(pairs))

In [24]:
V = len(train_vocab) + 2

for k in [1, 0.1, 0.01]:
    y_pred_lm = []
    for r in dev:
        pp_human = perplexity(r['tokens'], human_model, human_totals, V, k)
        pp_ai = perplexity(r['tokens'], ai_model, ai_totals, V, k)
        y_pred_lm.append(1 if pp_ai < pp_human else 0)
    print(f'k={k}  accuracy: {accuracy_score(y_dev, y_pred_lm):.3f}')
    print(confusion_matrix(y_dev, y_pred_lm))

k=1  accuracy: 0.916
[[ 507  132]
 [  19 1137]]
k=0.1  accuracy: 0.915
[[ 508  131]
 [  21 1135]]
k=0.01  accuracy: 0.916
[[ 510  129]
 [  21 1135]]


## Lesson 2: Questions from the slides

**1) How could we vectorize the texts for training a simple classifier?**
We used bag of words (CountVectorizer) and TF-IDF, and trained logistic regression on both. The vectorizer is only fit on train, so words that only show up in dev are ignored. The baseline (always guess AI) gets 0.644, so we mostly look at F1. Bag of words got accuracy 0.987 and TF-IDF got 0.969. We kept one-letter words like "i" and "a" by changing token_pattern, because they were different between humans and AI in lesson 1.

**2) How could we use n-gram LMs to distinguish human vs AI text?**
We trained one bigram model on human texts and one on AI texts. For each dev text we compute the perplexity under both models and pick the one with the lowest perplexity, because that model is the least surprised by the text. We need smoothing because many bigrams in dev were never seen in train. This got accuracy 0.916, but it leans towards AI: 132 of 639 human texts were guessed as AI.

**3) Should pre-processing for (1) be the same as (2)? Why or why not?**
Not necessarily. For the classifier it can be fine to remove stop words or normalize a lot, because it only looks at which words appear. An LM needs the word order and the small words like "in" and "the", otherwise it can't predict the next word. In our case we used the same cleaned tokens for both. We did not remove stop words, because function words like "i", "was" and "of" were different between humans and AI in lesson 1, so they are useful signals here.

## Lesson 3

In [25]:
%pip install spacy
import spacy.cli
spacy.cli.download("en_core_web_sm")


[notice] A new release of pip is available: 25.2 -> 26.2.2
[notice] To update, run: /opt/homebrew/opt/python@3.11/bin/python3.11 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 2.1 MB/s  0:00:06m0:00:0100:01m
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.



[notice] A new release of pip is available: 25.2 -> 26.2.2
[notice] To update, run: /opt/homebrew/opt/python@3.11/bin/python3.11 -m pip install --upgrade pip


In [26]:
nlp = spacy.load("en_core_web_sm")

doc = nlp(train[0]['text'][:300])

for token in doc:
    print(token.text, token.pos_, token.dep_, token.head.text, token.lemma_)

print("Verbs:", [token.text for token in doc if token.pos_ == "VERB"])

for ent in doc.ents:
    print(ent.text, ent.label_)

Duke PROPN compound Ellington Duke
Ellington PROPN nsubj revolutionized Ellington
, PUNCT punct Ellington ,
a DET det titan a
titan NOUN appos Ellington titan
of ADP prep titan of
jazz NOUN pobj of jazz
, PUNCT punct Ellington ,
revolutionized VERB ROOT revolutionized revolutionize
the DET det genre the
genre NOUN dobj revolutionized genre
through ADP prep revolutionized through
his PRON poss compositions his
innovative ADJ amod compositions innovative
compositions NOUN pobj through composition
, PUNCT punct revolutionized ,
showcasing VERB advcl revolutionized showcase
a DET det ability a
remarkable ADJ amod ability remarkable
ability NOUN dobj showcasing ability
to PART aux integrate to
integrate VERB acl ability integrate
voice NOUN dobj integrate voice
and CCONJ cc voice and
instrumental ADJ amod music instrumental
music NOUN conj voice music
. PUNCT punct revolutionized .
Among ADP prep was among
the DET det figures the
notable ADJ amod figures notable
figures NOUN pobj Among figu

In [27]:
def prepare(text):
    text = BeautifulSoup(text, "html.parser").get_text()
    return text[:2000]

def get_features(doc):
    words = [token for token in doc if token.is_alpha]
    sentences = list(doc.sents)
    n_words = max(len(words), 1)

    pos = [token.pos_ for token in doc]
    dep = [token.dep_ for token in doc]
    lemmas = [token.lemma_.lower() for token in words]
    ents = [ent.label_ for ent in doc.ents]

    distances = [abs(token.i - token.head.i) for token in words]

    style = {
        'avg_sentence_length': n_words / len(sentences),
        'avg_word_length': sum(len(token.text) for token in words) / n_words,
        'commas_per_word': len([t for t in doc if t.text == ',']) / n_words,
        'dashes_per_word': len([t for t in doc if t.text in ('—', '–')]) / n_words,
        'entities_per_word': len(ents) / n_words,
        'unique_lemmas': len(set(lemmas)) / n_words,
        'avg_dep_distance': sum(distances) / n_words,
    }

    return {
        'pos': ' '.join(pos),
        'dep': ' '.join(dep),
        'lemma': ' '.join(lemmas),
        'ner': ' '.join(ents),
        'style': style,
    }

In [28]:
start = time.time()
test = [get_features(doc) for doc in nlp.pipe([prepare(r['text']) for r in train[:200]])]
seconds = time.time() - start

print(round(seconds, 1), 'seconds for 200 texts')
print('About', round(seconds / 200 * len(train) / 60), 'minutes for all of train')
print(test[0]['style'])

9.7 seconds for 200 texts
About 19 minutes for all of train
{'avg_sentence_length': 26.545454545454547, 'avg_word_length': 5.678082191780822, 'commas_per_word': 0.07876712328767123, 'dashes_per_word': 0.0, 'entities_per_word': 0.03767123287671233, 'unique_lemmas': 0.5993150684931506, 'avg_dep_distance': 2.2226027397260273}


In [29]:
if os.path.exists('spacy_features.pkl'):
    with open('spacy_features.pkl', 'rb') as f:
        train_feats, dev_feats = pickle.load(f)
else:
    train_feats = [get_features(doc) for doc in nlp.pipe([prepare(r['text']) for r in train])]
    dev_feats = [get_features(doc) for doc in nlp.pipe([prepare(r['text']) for r in dev])]
    with open('spacy_features.pkl', 'wb') as f:
        pickle.dump((train_feats, dev_feats), f)

print(len(train_feats), len(dev_feats))

23707 1795


In [30]:
pos_vec = TfidfVectorizer(token_pattern=r'\S+', lowercase=False, ngram_range=(1, 3))
dep_vec = TfidfVectorizer(token_pattern=r'\S+', lowercase=False)
ner_vec = TfidfVectorizer(token_pattern=r'\S+', lowercase=False)
lemma_vec = CountVectorizer(token_pattern=r'\S+')

X_train_pos = pos_vec.fit_transform([f['pos'] for f in train_feats])
X_dev_pos = pos_vec.transform([f['pos'] for f in dev_feats])

X_train_dep = dep_vec.fit_transform([f['dep'] for f in train_feats])
X_dev_dep = dep_vec.transform([f['dep'] for f in dev_feats])

X_train_ner = ner_vec.fit_transform([f['ner'] for f in train_feats])
X_dev_ner = ner_vec.transform([f['ner'] for f in dev_feats])

X_train_lemma = lemma_vec.fit_transform([f['lemma'] for f in train_feats])
X_dev_lemma = lemma_vec.transform([f['lemma'] for f in dev_feats])

style_train = pd.DataFrame([f['style'] for f in train_feats])
style_dev = pd.DataFrame([f['style'] for f in dev_feats])
print(style_train.groupby(pd.Series(y_train)).mean())

scaler = StandardScaler()
X_train_style = csr_matrix(scaler.fit_transform(style_train))
X_dev_style = csr_matrix(scaler.transform(style_dev))

   avg_sentence_length  avg_word_length  commas_per_word  dashes_per_word  \
0            22.358854         4.386315         0.075102         0.001507   
1            22.312902         5.076073         0.079102         0.002183   

   entities_per_word  unique_lemmas  avg_dep_distance  
0           0.049662       0.517114          2.731133  
1           0.049218       0.575636          2.380389  


In [31]:
def train_and_test(name, X_tr, X_dv):
    clf = LogisticRegression(max_iter=3000)
    clf.fit(X_tr, y_train)
    pred = clf.predict(X_dv)
    print(name, '  accuracy:', round(accuracy_score(y_dev, pred), 3),
          '  macro F1:', round(f1_score(y_dev, pred, average='macro'), 3))
    return clf

train_and_test('BoW (lesson 2)', X_train, X_dev)
train_and_test('Lemmas', X_train_lemma, X_dev_lemma)
train_and_test('POS n-grams', X_train_pos, X_dev_pos)
train_and_test('Dependencies', X_train_dep, X_dev_dep)
train_and_test('NER', X_train_ner, X_dev_ner)
train_and_test('Style', X_train_style, X_dev_style)

X_train_ling = hstack([X_train_pos, X_train_dep, X_train_ner, X_train_style])
X_dev_ling = hstack([X_dev_pos, X_dev_dep, X_dev_ner, X_dev_style])
ling_model = train_and_test('All linguistic', X_train_ling, X_dev_ling)

X_train_all = hstack([X_train, X_train_ling])
X_dev_all = hstack([X_dev, X_dev_ling])
train_and_test('BoW + linguistic', X_train_all, X_dev_all)

BoW (lesson 2)   accuracy: 0.987   macro F1: 0.986
Lemmas   accuracy: 0.972   macro F1: 0.969
POS n-grams   accuracy: 0.934   macro F1: 0.927
Dependencies   accuracy: 0.877   macro F1: 0.865
NER   accuracy: 0.705   macro F1: 0.657
Style   accuracy: 0.888   macro F1: 0.877
All linguistic   accuracy: 0.942   macro F1: 0.936
BoW + linguistic   accuracy: 0.989   macro F1: 0.988


LogisticRegression(max_iter=3000)

**Style features (mean for human vs AI)**
Sentence length is almost the same (22.4 vs 22.3), so AI does not write longer sentences. The biggest difference is word length: AI uses longer words (5.1 vs 4.4 letters), which fits with the formal words we saw in lesson 2 like "significant" and "crucial". AI also has more unique lemmas per text (0.58 vs 0.52). This looks like the opposite of lesson 1, but in lesson 1 we counted unique words across all texts. So AI varies its words inside one text, but uses the same words again across texts. But this number goes down when a text gets longer, and some AI texts are shorter than 2000 characters, so part of the difference might just come from text length. Humans have a longer average dependency distance (2.7 vs 2.4), so their sentences are less neatly built.

**Results**
- BoW (lesson 2): macro F1 0.986
- Lemmas: 0.969
- POS n-grams: 0.927
- Dependencies: 0.865
- NER: 0.657
- Style (only 7 numbers): 0.877
- All linguistic (no words): 0.936
- BoW + linguistic: 0.988

Without knowing any words, the linguistic features still get 0.936, so the difference between human and AI is not only what is written but also how. POS n-grams work best on their own. NER is weak, so names don't say much about who wrote the text. Lemmas are a bit worse than BoW, maybe because lemmatization removes information like "was" vs "is". Adding linguistic features to BoW only helps a little, because BoW is already very high.
Note: the spaCy features only use the first 2000 characters of each text, while BoW uses the full text, so the comparison is not completely fair.

In [32]:
names = (list('pos: ' + pos_vec.get_feature_names_out())
         + list('dep: ' + dep_vec.get_feature_names_out())
         + list('ner: ' + ner_vec.get_feature_names_out())
         + list('style: ' + style_train.columns))

coefs = ling_model.coef_[0]
print('Most AI:')
for i in np.argsort(coefs)[-15:][::-1]:
    print('  ', names[i], round(coefs[i], 2))

print('\nMost human:')
for i in np.argsort(coefs)[:15]:
    print('  ', names[i], round(coefs[i], 2))

Most AI:
   dep: poss 6.74
   pos: PUNCT PRON NOUN 3.91
   pos: PUNCT PUNCT 3.74
   pos: PUNCT PUNCT PROPN 3.51
   pos: NOUN CCONJ 3.32
   pos: PUNCT PUNCT PRON 3.16
   pos: NOUN PUNCT PUNCT 3.05
   pos: ADP NOUN CCONJ 2.89
   dep: det 2.89
   pos: DET ADJ NOUN 2.85
   pos: NOUN PUNCT VERB 2.75
   pos: PUNCT PRON 2.74
   dep: appos 2.73
   pos: PROPN SPACE 2.72
   dep: nsubj 2.7

Most human:
   dep: conj -7.92
   pos: PUNCT CCONJ VERB -4.57
   pos: SPACE PUNCT -4.38
   pos: PUNCT SPACE PUNCT -4.07
   pos: NOUN PUNCT NOUN -4.01
   pos: PUNCT ADP -3.81
   pos: PUNCT CCONJ -3.36
   pos: PUNCT -3.26
   pos: ADP -3.19
   dep: advmod -3.04
   pos: PUNCT NOUN -2.99
   pos: PUNCT ADJ -2.94
   dep: expl -2.9
   dep: auxpass -2.73
   pos: PROPN PUNCT SPACE -2.62


**Which linguistic features does the model use?**
The strongest AI feature is possessives (dep: poss), like "its significance" or "his innovative compositions". AI also uses more appositions (appos), like "Duke Ellington, a titan of jazz", and DET ADJ NOUN like "a remarkable ability". So AI writes in a describing and explaining style.

Both groups use "and", but in different ways. AI often joins nouns (NOUN CCONJ, like "innovation and creativity"). Humans use more coordination in general (dep: conj), and often join clauses with a comma and a verb (PUNCT CCONJ VERB, like ", and walked out"). This fits with lesson 1, where humans wrote more like a story in past tense. Humans also use more "there was" (expl) and passive (auxpass). Passive is surprising, because formal text is often thought to use more passive, but it might come from human news texts.

We should be careful with SPACE and PUNCT PUNCT. SPACE is extra spaces and line breaks, so it is about how the texts are formatted in the dataset and not about language. It could be a shortcut like the Chinese characters in lesson 1.
The style features are not on the list, but their weights can't be compared directly with the TF-IDF features, because they are scaled differently.

**Limitation**
The dev texts are new, so there is no data contamination. But dev comes from the same AI models and the same three genres as train. So our results show how well the models work on texts that look like the training data, not how well they would work on a new AI model or a different kind of text. In real use the results would probably be lower.

## Lesson 3: Questions from the slides

**1) If we want to use NER, POS tags and dependency parsing features, how should our pre-processing pipeline go? Where could normalization occur?**
Clean (only remove HTML) → spaCy (tokenize, split sentences, POS, dependency parsing, NER, lemma) → normalize (lowercase lemmas) → feature vectors.
The normalization has to happen after parsing, not before. spaCy needs capital letters to find names (NER), punctuation to find sentences, and small words like "the" and "of" to build the dependency tree. If we used clean_text from lesson 1 first, the parser and NER would get much worse input.

**2a) Why don't we count POS tags?**
Raw counts mostly measure how long the text is, because a longer text just has more nouns and verbs. And a single tag like NOUN doesn't say much on its own. So we used TF-IDF, which turns them into proportions, and POS n-grams like "DET ADJ NOUN", which say something about how the sentences are built. POS n-grams alone got macro F1 0.927.

**2b) Why might we count syntactic dependencies?**
Dependencies show how the sentences are built, not what they are about. So they capture style, which could work across topics where bag of words might just learn the topic. Dependencies alone got macro F1 0.865, and some of the strongest features were dependencies: possessives (poss) for AI and coordination (conj) for humans.

**2c) How else might we use dependency parsing?**
We used it for a style feature: the average distance between a word and its head in the tree. Humans had a longer average distance (2.7 vs 2.4). You could also use it to find passive sentences or to pull out subject-verb pairs.

## Lektion 4


In [33]:
glove = api.load("glove-wiki-gigaword-100")

print(len(glove.key_to_index), 'words in GloVe')
print(glove['king'][:10])

400000 words in GloVe
[-0.32307 -0.87616  0.21977  0.25268  0.22976  0.7388  -0.37954 -0.35307
 -0.84369 -1.1113 ]


In [34]:
print(glove.most_similar('king', topn=5))
print(glove.similarity('car', 'automobile'))
print(glove.similarity('car', 'banana'))
print(glove.doesnt_match(['breakfast', 'lunch', 'dinner', 'car']))
print(glove.most_similar(positive=['king', 'woman'], negative=['man'], topn=3))

print(glove.most_similar('crucial', topn=5))
print(glove.most_similar('significant', topn=5))

[('prince', 0.7682329416275024), ('queen', 0.7507689595222473), ('son', 0.7020887136459351), ('brother', 0.6985775828361511), ('monarch', 0.6977890729904175)]
0.68319416
0.14861871
car
[('queen', 0.7698541283607483), ('monarch', 0.6843381524085999), ('throne', 0.6755736470222473)]
[('vital', 0.8405664563179016), ('key', 0.8310879468917847), ('pivotal', 0.7820190191268921), ('important', 0.7793831825256348), ('decisive', 0.7502336502075195)]
[('substantial', 0.887603223323822), ('important', 0.8133977651596069), ('major', 0.7773129343986511), ('considerable', 0.7689317464828491), ('impact', 0.7590166926383972)]


In [35]:
total = sum(all_Vocab.values())
known = sum(count for word, count in all_Vocab.items() if word in glove.key_to_index)
print('Share of tokens GloVe knows:', round(known / total * 100, 1), '%')

unique_known = sum(1 for word in all_Vocab if word in glove.key_to_index)
print('Share of unique words GloVe knows:', round(unique_known / len(all_Vocab) * 100, 1), '%')

Share of tokens GloVe knows: 98.7 %
Share of unique words GloVe knows: 51.6 %


In [36]:
def doc_vector(tokens, kv):
    vectors = [kv[t] for t in tokens if t in kv.key_to_index]
    if len(vectors) == 0:
        return np.zeros(kv.vector_size)
    return np.mean(vectors, axis=0)

def doc_vector_mean_max(tokens, kv):
    vectors = [kv[t] for t in tokens if t in kv.key_to_index]
    if len(vectors) == 0:
        return np.zeros(kv.vector_size * 2)
    return np.concatenate([np.mean(vectors, axis=0), np.max(vectors, axis=0)])

X_train_glove = np.array([doc_vector(r['tokens'], glove) for r in train])
X_dev_glove = np.array([doc_vector(r['tokens'], glove) for r in dev])

X_train_glove_mm = np.array([doc_vector_mean_max(r['tokens'], glove) for r in train])
X_dev_glove_mm = np.array([doc_vector_mean_max(r['tokens'], glove) for r in dev])

print(X_train_glove.shape, X_train_glove_mm.shape)

(23707, 100) (23707, 200)


In [37]:
scaler_glove = StandardScaler()
X_train_glove_s = scaler_glove.fit_transform(X_train_glove)
X_dev_glove_s = scaler_glove.transform(X_dev_glove)

scaler_mm = StandardScaler()
X_train_mm_s = scaler_mm.fit_transform(X_train_glove_mm)
X_dev_mm_s = scaler_mm.transform(X_dev_glove_mm)

_ = train_and_test('BoW (lesson 2)', X_train, X_dev)
_ = train_and_test('GloVe mean', X_train_glove_s, X_dev_glove_s)
_ = train_and_test('GloVe mean + max', X_train_mm_s, X_dev_mm_s)

BoW (lesson 2)   accuracy: 0.987   macro F1: 0.986
GloVe mean   accuracy: 0.954   macro F1: 0.95
GloVe mean + max   accuracy: 0.963   macro F1: 0.96


In [40]:
w2v = Word2Vec(sentences=[r['tokens'] for r in train], vector_size=100,
               window=5, min_count=5, workers=4)

print(w2v.wv.most_similar('crucial', topn=5))
print(w2v.wv.most_similar('significant', topn=5))

X_train_w2v = np.array([doc_vector(r['tokens'], w2v.wv) for r in train])
X_dev_w2v = np.array([doc_vector(r['tokens'], w2v.wv) for r in dev])

scaler_w2v = StandardScaler()
_ = train_and_test('Own Word2Vec mean',
                   scaler_w2v.fit_transform(X_train_w2v),
                   scaler_w2v.transform(X_dev_w2v))

Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'


[('vital', 0.8464141488075256), ('essential', 0.8273950815200806), ('critical', 0.7892715334892273), ('paramount', 0.6951521039009094), ('key', 0.6861653327941895)]
[('substantial', 0.7286503314971924), ('notable', 0.6634335517883301), ('pivotal', 0.6514496803283691), ('critical', 0.6109731197357178), ('crucial', 0.5991825461387634)]
Own Word2Vec mean   accuracy: 0.971   macro F1: 0.968


**Results lesson 4**
- BoW (lesson 2): macro F1 0.986 (139,096 numbers per text)
- Own Word2Vec mean: 0.968 (100 numbers)
- GloVe mean + max: 0.96 (200 numbers)
- GloVe mean: 0.95 (100 numbers)
- All linguistic (lesson 3): 0.936

Embeddings get almost the same result as bag of words with only 100 numbers per text instead of 139,096. Mean + max is a bit better than mean alone, so the max catches something the average misses.
Our own Word2Vec is better than GloVe, even though GloVe is trained on much more text. Our model is trained on the same kind of text it has to classify, while GloVe is trained on Wikipedia and news from 2014, before modern AI.
The nearest neighbors of "crucial" in our own model are "vital", "essential" and "critical", and for "significant" they are "substantial", "notable" and "pivotal". So the model has learned that these words mean the same. They are also all typical AI words. So when we average all the words in a text, AI texts end up in the same area, because AI uses many words from the same formal group. Embeddings don't catch the exact word, but they catch the style.
The "Exception ignored" lines from gensim are only warnings and don't affect the result.